# RAG Agent using langgraph 

### Fetching the document to use in our `Rag` system

In [23]:
import getpass
import os


def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}:")


_set_env("OPENAI_API_KEY")

In [9]:
import bs4 
import requests
from langchain_core.documents import Document


def load_web_page(url: str, bs_kwrg : dict | None = None) -> list[Document]:
    response = requests.get(url,timeout=30)
    response.raise_for_status()
    soup = bs4.BeautifulSoup(response.text,"html.parser",**(bs_kwrg or {}))
    
    return [Document(page_content=soup.get_text(),metadata ={"source":url})]


urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
]

docs = [load_web_page(url) for url in urls]

### Chunking the text


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 200,
    chunk_overlap = 50,
)

doc_splits = text_splitter.split_documents(docs_list)

In [11]:
len(doc_splits) # total Chunks


239

### creating the Retriveal tool 

#### using an in-memory vector store and OpenAi embeddings 

In [27]:
%pip install -U langchain-ollama

Defaulting to user installation because normal site-packages is not writeable
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:

In [28]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from functools import lru_cache

embedding = OllamaEmbeddings(
   model = "nomic-embed-text"
)
@lru_cache(maxsize=1)
def _get_retriver():
    vector_store = InMemoryVectorStore.from_documents(
       documents= doc_splits,
       embedding= embedding,
        
    )
    
    return vector_store.as_retriever()
    

creating an retriver tool using the `@tool` decorator

In [29]:
from langchain.tools import tool

@tool
def retrive_blog_posts(query:str) -> str:
    """search and return the most sementic chunk from the urls """
    retriver = _get_retriver()
    retrived_doc = retriver.invoke(query)
    return "\n\n".join([doc.page_content for doc in retrived_doc])

retriver_tool = retrive_blog_posts

In [30]:
retriver_tool.invoke({"query": "types of reward hacking"})

"In-Context Reward Hacking#\n\nReward Tampering (Everitt et al. 2019) is a form of reward hacking behavior where the agent interferes with the reward function itself, causing the observed reward to no longer accurately represent the intended goal. In reward tampering, the model modifies its reward mechanism either by directly manipulating the implementation of the reward function or by indirectly altering the environmental information used as input for the reward function.\n(Note: Some work defines reward tampering as a distinct category of misalignment behavior from reward hacking. But I consider reward hacking as a broader concept here.)\nAt a high level, reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering.\n\nReward hacking (Amodei et al., 2016)\nReward corruption (Everitt et al., 2017)\nReward tampering (Everitt et al. 2019)\nSpecification gaming (Krakovna et al., 2020)\nObjective robustness (Koch et al. 2021)\nGoal misgenera